# Laboratorio #8

* Josue Say - 228801
* Flavio Galán - 22386

## Repositorio

- [Enlace](https://github.com/JosueSay/labs-ds/tree/main/lab8)

## Librerías

In [44]:
import pandas as pd
import pyreadstat
import os
import re
import glob
import web_scrapping
import subprocess
from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, collect_set, sort_array

## Constantes

In [45]:
DATA_BASE = "/home/jovyan/work/data"
RAW_BASE = os.path.join(DATA_BASE, "raw")
CACHE_BASE = "/home/jovyan/work/cache"

CACHE_WEB_SCRAPPING = "cache_web_scrapping.txt"
CACHE_TRANSFORM_SAV = "cache_transform_sav.txt"

# Aseguramos que existan las carpetas
os.makedirs(DATA_BASE, exist_ok=True)
os.makedirs(CACHE_BASE, exist_ok=True)

In [46]:
def printTree(root, prefix=""):
    files = os.listdir(root)
    for i, f in enumerate(files):
        path = os.path.join(root, f)
        connector = "└── " if i == len(files) - 1 else "├── "
        print(prefix + connector + f)
        if os.path.isdir(path):
            extension = "    " if i == len(files) - 1 else "│   "
            printTree(path, prefix + extension)

In [47]:
def convertSavToXlsx(inputPath, outputPath):
    df, meta = pyreadstat.read_sav(inputPath)
    df.to_excel(outputPath, index=False)
    print(f"Convertido: {inputPath} -> {outputPath}")

In [48]:
def runWebScrapping():
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_WEB_SCRAPPING)

    if os.path.exists(cacheFilePath):
        print("Ya se ha ejecutado el proceso de web_scrapping.")
        return

    # Ejecutar web_scrapping.py desde el notebook
    scriptPath = os.path.join(os.getcwd(), "web_scrapping.py")
    if not os.path.exists(scriptPath):
        print(f"No se encontró {scriptPath}")
        return

    print("Ejecutando web_scrapping.py ...")
    result = subprocess.run(["python", scriptPath], capture_output=True, text=True)

    # Mostrar salida en notebook
    print(result.stdout)
    if result.stderr:
        print("Errores:", result.stderr)

In [49]:
def processSavFiles():
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_TRANSFORM_SAV)

    # Si el archivo de caché existe, saltamos la conversión
    if os.path.exists(cacheFilePath):
        print("Archivo de caché encontrado. Saltando conversión de .sav a .xlsx.")
        return

    # Recorremos los archivos .sav
    for root, dirs, files in os.walk(RAW_BASE):
        for f in files:
            if f.endswith(".sav"):
                inputFile = os.path.join(root, f)
                outputFile = inputFile.replace(".sav", ".xlsx")
                convertSavToXlsx(inputFile, outputFile)

    # Creamos el archivo de caché al final
    with open(cacheFilePath, "w") as cacheFile:
        cacheFile.write("Transformación de .sav completada.\n")
    print(f"Caché creado: {cacheFilePath}")

In [50]:
runWebScrapping()

Ejecutando web_scrapping.py ...

=== Año 2013 ===
GET https://www.ine.gob.gt/bdatos_cargar.php?anio=2013&categoria=140&periodo=1&dire=https://www.ine.gob.gt/sistema/
	• hechos: 0NGWpC89u7XxdYqecRmMtNfW9uU9Npxc.sav -> /home/jovyan/work/data/raw/2013/0NGWpC89u7XxdYqecRmMtNfW9uU9Npxc.sav
	• vehiculos: Z4mQf2vBSUwZH8r99seywakg93YtNzkZ.sav -> /home/jovyan/work/data/raw/2013/Z4mQf2vBSUwZH8r99seywakg93YtNzkZ.sav
	• fallecidos_lesionados: b0V72Mghnp6ujvMU8XlLELO68wFOodoy.sav -> /home/jovyan/work/data/raw/2013/b0V72Mghnp6ujvMU8XlLELO68wFOodoy.sav
	✓ Descargados 3 archivo(s) para 2013

=== Año 2014 ===
GET https://www.ine.gob.gt/bdatos_cargar.php?anio=2014&categoria=140&periodo=1&dire=https://www.ine.gob.gt/sistema/
	• hechos: DX2BmYU5m4JfPRhrFHwDRDEs49V7fN5I.sav -> /home/jovyan/work/data/raw/2014/DX2BmYU5m4JfPRhrFHwDRDEs49V7fN5I.sav
	• vehiculos: MVtI7adfQzZWF5uefXZ4xmZVG0vRik3S.sav -> /home/jovyan/work/data/raw/2014/MVtI7adfQzZWF5uefXZ4xmZVG0vRik3S.sav
	• fallecidos_lesionados: Qr9erLDhFszDZ3A

In [51]:
printTree(RAW_BASE)

├── 2013
│   ├── 0NGWpC89u7XxdYqecRmMtNfW9uU9Npxc.sav
│   ├── b0V72Mghnp6ujvMU8XlLELO68wFOodoy.sav
│   └── Z4mQf2vBSUwZH8r99seywakg93YtNzkZ.sav
├── 2014
│   ├── DX2BmYU5m4JfPRhrFHwDRDEs49V7fN5I.sav
│   ├── MVtI7adfQzZWF5uefXZ4xmZVG0vRik3S.sav
│   └── Qr9erLDhFszDZ3AgrA5NDIVCs9KURG0w.sav
├── 2015
│   ├── fflzORJDQM0tVur8UvU6fz5dR51kogHr.xlsx
│   ├── iZwkJLBrcyCmachhwmugKt4pf6cK8kRg.xlsx
│   └── jPrxJop87uz1zGIf94EvWdACDtMLxREE.xlsx
├── 2016
│   ├── aE63D8ky7MoFhXG3MgBOYfWXBzsEFBGD.xlsx
│   ├── DX2BmYU5m4JfPRhrFHwDRDEs49V7fN5I.xlsx
│   └── MVtI7adfQzZWF5uefXZ4xmZVG0vRik3S.xlsx
├── 2017
│   ├── 2018060193914Nyto5KpgXeUsKGoT4SpRknBumA8etDe4.xlsx
│   ├── 2018060194026zskZfNalr2em0qLC5Wn6bxC1mBim617t.xlsx
│   └── 201806191110218FciGNFOtT2FJnkTOS0pTzPcDOW8FpLB.xlsx
├── 2018
│   ├── 20190606220307YRSvO7OHib0rQxKDCTAl2fkXMl05g9Uz.xlsx
│   ├── 20190606220636xaPevkVgXaNin0L0ZmXiN4fm18JAFoLG.xlsx
│   └── 20190610171521QZwxwRe5OLADYzMpAncpW3yR4defMHqN.xlsx
├── 2019
│   ├── 20200519180527eLWpCfULaJg

In [52]:
processSavFiles()

Convertido: /home/jovyan/work/data/raw/2013/0NGWpC89u7XxdYqecRmMtNfW9uU9Npxc.sav -> /home/jovyan/work/data/raw/2013/0NGWpC89u7XxdYqecRmMtNfW9uU9Npxc.xlsx
Convertido: /home/jovyan/work/data/raw/2013/b0V72Mghnp6ujvMU8XlLELO68wFOodoy.sav -> /home/jovyan/work/data/raw/2013/b0V72Mghnp6ujvMU8XlLELO68wFOodoy.xlsx
Convertido: /home/jovyan/work/data/raw/2013/Z4mQf2vBSUwZH8r99seywakg93YtNzkZ.sav -> /home/jovyan/work/data/raw/2013/Z4mQf2vBSUwZH8r99seywakg93YtNzkZ.xlsx
Convertido: /home/jovyan/work/data/raw/2014/DX2BmYU5m4JfPRhrFHwDRDEs49V7fN5I.sav -> /home/jovyan/work/data/raw/2014/DX2BmYU5m4JfPRhrFHwDRDEs49V7fN5I.xlsx
Convertido: /home/jovyan/work/data/raw/2014/MVtI7adfQzZWF5uefXZ4xmZVG0vRik3S.sav -> /home/jovyan/work/data/raw/2014/MVtI7adfQzZWF5uefXZ4xmZVG0vRik3S.xlsx
Convertido: /home/jovyan/work/data/raw/2014/Qr9erLDhFszDZ3AgrA5NDIVCs9KURG0w.sav -> /home/jovyan/work/data/raw/2014/Qr9erLDhFszDZ3AgrA5NDIVCs9KURG0w.xlsx
Convertido: /home/jovyan/work/data/raw/2021/2022062310154FYBQhYtMxyPU3C9wbQb

In [2]:


spark = SparkSession.builder.appName("Lab8").getOrCreate()

      # notebooks/data/raw
         # notebooks/data
CSV_DIRS = {
    "hechos": f"{OUT_BASE}/hechos",
    "vehiculos": f"{OUT_BASE}/vehiculos",
    "fallecidos_lesionados": f"{OUT_BASE}/fallecidos_lesionados",
}
for d in CSV_DIRS.values():
    os.makedirs(d, exist_ok=True)

YEARS = range(2013, 2024)  # 2013–2023

def norm_cols(df):
    df.columns = (df.columns.str.strip().str.lower()
                  .str.normalize('NFKD').str.encode('ascii','ignore').str.decode('ascii')
                  .str.replace(r'\s+','_',regex=True))
    # map a 'anio'
    for c in ["anio","año","ano","year"]:
        if c in df.columns:
            df = df.rename(columns={c:"anio"})
            break
    return df

# Heurística por columnas para clasificar archivos (porque los nombres están “hash”)
def guess_category(cols_lower_set):
    txt = " ".join(cols_lower_set)
    if re.search(r"vehicul|conductor|color|modelo|placa", txt): return "vehiculos"
    if re.search(r"lesionad|fallecid|edad|sexo", txt): return "fallecidos_lesionados"
    # por descarte:
    return "hechos"

## Convertir a CSV (XLSX / SAV)

In [3]:
def to_csv_per_year():
    for y in YEARS:
        year_dir = os.path.join(RAW_BASE, str(y))
        if not os.path.isdir(year_dir): 
            print(f"[{y}] sin carpeta, skip"); 
            continue
        files = glob.glob(os.path.join(year_dir, "*.xlsx")) + glob.glob(os.path.join(year_dir, "*.sav"))
        if not files:
            print(f"[{y}] sin archivos, skip"); 
            continue

        # leer cada archivo, clasificar por columnas y guardar CSV {cat}_{y}.csv
        seen_cat = set()
        for f in files:
            if f.endswith(".xlsx"):
                df = pd.read_excel(f, sheet_name=0, engine="openpyxl")
            else:
                df, _meta = pyreadstat.read_sav(f, apply_value_formats=True)

            df = norm_cols(pd.DataFrame(df))
            df["anio"] = y
            cat = guess_category(set(df.columns))

            # guardar
            out_path = os.path.join(CSV_DIRS[cat], f"{cat}_{y}.csv")
            df.to_csv(out_path, index=False, encoding="utf-8")
            seen_cat.add(cat)
            print(f"[{y}] {os.path.basename(f)} → {cat}_{y}.csv")

        # validación rápida
        for cat in ["hechos","vehiculos","fallecidos_lesionados"]:
            if not os.path.exists(os.path.join(CSV_DIRS[cat], f"{cat}_{y}.csv")):
                print(f"[WARN {y}] faltó {cat}")

to_csv_per_year()
print("✅ CSV por año listo en:", CSV_DIRS)


[2013] 0NGWpC89u7XxdYqecRmMtNfW9uU9Npxc.sav → vehiculos_2013.csv
[2013] b0V72Mghnp6ujvMU8XlLELO68wFOodoy.sav → vehiculos_2013.csv
[2013] Z4mQf2vBSUwZH8r99seywakg93YtNzkZ.sav → vehiculos_2013.csv
[WARN 2013] faltó hechos
[WARN 2013] faltó fallecidos_lesionados
[2014] DX2BmYU5m4JfPRhrFHwDRDEs49V7fN5I.sav → vehiculos_2014.csv
[2014] MVtI7adfQzZWF5uefXZ4xmZVG0vRik3S.sav → vehiculos_2014.csv
[2014] Qr9erLDhFszDZ3AgrA5NDIVCs9KURG0w.sav → vehiculos_2014.csv
[WARN 2014] faltó hechos
[WARN 2014] faltó fallecidos_lesionados
[2015] fflzORJDQM0tVur8UvU6fz5dR51kogHr.xlsx → vehiculos_2015.csv
[2015] iZwkJLBrcyCmachhwmugKt4pf6cK8kRg.xlsx → vehiculos_2015.csv
[2015] jPrxJop87uz1zGIf94EvWdACDtMLxREE.xlsx → vehiculos_2015.csv
[WARN 2015] faltó hechos
[WARN 2015] faltó fallecidos_lesionados
[2016] aE63D8ky7MoFhXG3MgBOYfWXBzsEFBGD.xlsx → vehiculos_2016.csv
[2016] DX2BmYU5m4JfPRhrFHwDRDEs49V7fN5I.xlsx → vehiculos_2016.csv
[2016] MVtI7adfQzZWF5uefXZ4xmZVG0vRik3S.xlsx → vehiculos_2016.csv
[WARN 2016] faltó h

## Cargar en Spark

In [4]:
df_hechos = spark.read.csv(f"{CSV_DIRS['hechos']}/*.csv", header=True, inferSchema=True)
df_veh = spark.read.csv(f"{CSV_DIRS['vehiculos']}/*.csv", header=True, inferSchema=True)
df_fall = spark.read.csv(f"{CSV_DIRS['fallecidos_lesionados']}/*.csv", header=True, inferSchema=True)

# (opcional) cache
df_hechos.cache(); df_veh.cache(); df_fall.cache();

print("hechos:", df_hechos.count(), "vehiculos:", df_veh.count(), "fall/les:", df_fall.count())
df_hechos.printSchema()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/work/data/hechos/*.csv.

## Conteos

In [ ]:
for name, df in [("hechos", df_hechos), ("vehiculos", df_veh), ("fallecidos_lesionados", df_fall)]:
    print(f"\n=== {name.upper()} ===")
    print("rows:", df.count())
    df.show(5, truncate=False)
    # columnas numéricas comunes (ajusta a tus nombres reales)
    num_cols = [c for c,t in df.dtypes if t in ("int","bigint","double","float")]
    if num_cols:
        df.select(*[col(c) for c in num_cols]).describe().show()
        df.select(*[col(c) for c in num_cols]).summary().show()

## Años disponibles

In [ ]:
for name, df in [("hechos", df_hechos), ("vehiculos", df_veh), ("fallecidos_lesionados", df_fall)]:
    print(f"\nAños en {name}:")
    df.select("anio").distinct().orderBy("anio").show(200)

## Valores distintos de tipo de accidente

In [ ]:
for colname in ["tipo_accidente","tipo_de_accidente","tipo"]:
    if colname in df_hechos.columns:
        print(f"Valores distintos de {colname}:")
        df_hechos.select(colname).distinct().orderBy(colname).show(200, truncate=False)
        break


## Departamentos únicos

In [ ]:
posibles = ["departamento","depto","dept"]
def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns: return c
    return None

cols = {
    "hechos": first_existing(df_hechos, posibles),
    "vehiculos": first_existing(df_veh, posibles),
    "fallecidos_lesionados": first_existing(df_fall, posibles),
}
for name, df in [("hechos", df_hechos), ("vehiculos", df_veh), ("fallecidos_lesionados", df_fall)]:
    c = cols[name]
    if c:
        print(f"\nDepartamentos únicos en {name} (col: {c}):")
        df.select(c).distinct().orderBy(c).show(200, truncate=False)
    else:
        print(f"[WARN] No encontré columna departamento en {name}.")